# EXP03 — Investigação dos alertas de 2020

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. O que verificamos?

A classificação informa uma pontuação, mas não identifica automaticamente a função do contrato. Este notebook lê uma lista de alertas produzida pelo EXP01, organiza candidatos para conferência e **não cria rótulos negativos**.

In [ ]:
import pandas as pd
from pathlib import Path

arquivos = sorted((RAIZ / "results" / "generated" / "EXP01").glob("*/alertas.csv"))
if not arquivos:
    raise FileNotFoundError("Execute primeiro o EXP01 e gere alertas.csv.")
alertas = pd.read_csv(arquivos[-1])
colunas = [c for c in ["recipient", "n_tx", "score", "tornado"] if c in alertas]
print("Alertas encontrados:", len(alertas))
display(alertas[colunas].sort_values("score", ascending=False))

## 3. Ficha de conferência

Complete as colunas com evidências da data histórica antes de atribuir qualquer rótulo. A ausência de documentação não prova que uma carteira não seja mixer.

In [ ]:
fichas = alertas[["recipient", "score"]].copy()
fichas["fonte_onchain"] = ""
fichas["tipo_historico"] = "indeterminado"
fichas["justificativa"] = ""
saida = RAIZ / "results" / "generated" / "EXP03"
saida.mkdir(parents=True, exist_ok=True)
fichas.to_csv(saida / "fichas_pendentes.csv", index=False)
print("Ficha criada:", saida / "fichas_pendentes.csv")

## 4. Limitação

Esta é uma ferramenta de organização de auditoria, não um procedimento automático de confirmação funcional. Os relatos EXP-01B a EXP-01I permanecem como fontes históricas.